# ProLM patient embeddings on SynthTAN, trained inside cross-validation folds

This notebook runs `cv_embeddings.py` (same folder) on one SynthTAN cohort and checks the result.
It replaces the earlier single 50/50 split, which had four problems:

1. **Labels were mispaired during training.** The training loader shuffled the patients, but the loop ignored the
   loader's labels and used `labels[train_idx]` in the original order, so expression rows and labels did not match
   (re-shuffled every epoch).
2. **Evaluation did not score the six tasks.** `evaluate()` took a softmax over the six outputs, used output 1
   (MOT1_A) and compared it with the `DOM` labels only (final AUROC 0.49; 0.55 at the selected epoch). Six
   independent 0/1 tasks need one sigmoid per task, each scored against its own labels.
3. **The epoch was chosen on the test half**, and one model embedded all 200 patients, after training on the
   labels of the training half. For late fusion the embeddings must come from a model trained inside each fold.
4. **One optimizer step per epoch:** 100 training patients with a batch size of 128 gave 20 steps in total.
   `test_loss` also compared the last *training* batch's outputs with the test labels.

Each fold's outputs include `<run>_fold<k>_train.txt` / `<run>_fold<k>_test.txt` (one id per line), the format
of `--train-individuals` in the genome steps (`1_build_graph.py`, `3_node_embeddings.py`,
`4a_finetune_evo2_embeddings.py`) and of `--train-individuals` / `--test-individuals` in `5_validate_evo2.py`, so
the genome embeddings can be built on the same folds and joined to `patient_embeddings.csv` on `eid`.

In [ ]:
! ls /home/shadeform/Data

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd

data_dir = Path('/home/shadeform/Data')       # expression.csv, labels_<TASK>.csv, protein_embeddings.npz
output_dir = Path('/home/shadeform/Output/prolm_cv')
script = Path('cv_embeddings.py')              # run the notebook from proteomics_emb/, or give the full path

## Train ProLM inside 5 folds

All six labels are learned at once (one sigmoid output each). Within each fold, 20% of the training patients
are held out to choose the epoch (early stopping on their loss); the test fold is not used until the end.
To reuse folds made elsewhere (for example for the genome side), add `--folds-csv folds.csv` (`eid, fold`).

In [ ]:
cmd = [sys.executable, str(script),
       '--expression-csv', str(data_dir / 'expression.csv'),
       '--labels-dir', str(data_dir),
       '--protein-embeddings', str(data_dir / 'protein_embeddings.npz'),
       '--output-dir', str(output_dir),
       '--device', 'cuda',
       '--n-folds', '5', '--hidden-size', '32', '--num-layers', '1', '--chunk-size', '64',
       '--batch-size', '16', '--max-epochs', '100', '--patience', '15', '--learning-rate', '1e-3']
subprocess.run(cmd, check=True)

## Check the protein-only results

Test-fold AUROC per task over folds (ProLM's own head). On SynthTAN v0.1 one would expect MOT1_A (strong tracker
protein on the region) to be the easiest and NULL_INDEP_A (no protein responds) to be near 0.5. A null AUROC
clearly above 0.5 calls for a check (e.g. rerun with permuted labels), not a conclusion. SUB2 has 24 positives at
n = 200 only; in the smaller cohorts its per-fold values are undefined or unstable.

In [ ]:
summary = pd.read_csv(output_dir / 'summary.csv')
summary.groupby('task', sort=False).agg(mean_auroc=('test_auroc', 'mean'), sd_auroc=('test_auroc', 'std'),
                                        mean_ap=('test_ap', 'mean'), folds_with_auroc=('test_auroc', 'count')).round(3)

## Embeddings for late fusion

One file per fold. Train the downstream model on the fold's training patients and test on its test patients;
never mix embeddings from different folds (each fold's model has its own vector space) or cohorts (eids repeat).
Use the same downstream learner and training rows for genome only, protein only and genome + protein.

In [ ]:
import json
cfg = json.loads((output_dir / 'config.json').read_text())
run = cfg['fingerprint']['run_name']
folds = pd.read_csv(output_dir / 'folds.csv', index_col='eid')['fold']
expression_ids = pd.read_csv(data_dir / 'expression.csv', usecols=['eid'], dtype=str)['eid']
assert folds.index.is_unique and set(folds.index) == set(expression_ids)
for k in sorted(folds.unique()):
    fdir = output_dir / f'fold{k}'
    emb = pd.read_csv(fdir / 'patient_embeddings.csv', index_col='eid')
    train_ids = (fdir / f'{run}_fold{k}_train.txt').read_text().split()
    test_ids = (fdir / f'{run}_fold{k}_test.txt').read_text().split()
    assert emb.index.is_unique and len(set(train_ids)) == len(train_ids) and len(set(test_ids)) == len(test_ids)
    assert not set(train_ids) & set(test_ids)
    assert set(train_ids) | set(test_ids) == set(emb.index) == set(folds.index)
    assert set(test_ids) == set(folds.index[folds == k])
    print(f'fold {k}: embeddings {emb.shape}, {len(train_ids)} training and {len(test_ids)} test patients')
emb.head()